In [11]:
import numpy as np#这里需要分X的代码
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 加载数据
data = fetch_california_housing()

X = data.data          # 特征矩阵
y = data.target        # 目标值（房价）

feature_names = data.feature_names

# 转为 DataFrame（推荐，方便后续操作）
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="MedHouseVal")

print(X.head())
print(y.head())
print("Shape of X:", X.shape)

# ===== 处理缺失值 =====
print("Missing values before cleaning:")
print(X.isnull().sum())

# 删除含缺失值的行（如果有）
X = X.dropna()
y = y.loc[X.index]   # 保持 X 和 y 对齐

print("Shape after removing missing values:", X.shape)

from sklearn.model_selection import train_test_split#这里开始
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 先分出 test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=26
)
#X_train_val → 80% 的特征，X_test → 20% 的特征，y_train_val → 80% 的标签，y_test → 20% 的标签

# 从X_train_val, y_train_val里再分 validation（分成X_train, X_val, y_train, y_val）
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1, random_state=26
)#这里的0.1是80%里的10%


# 在标准化前，先保存 raw（未标准化）版本：后面kNN需要用原始经纬度
X_train_raw = X_train.copy()
X_val_raw   = X_val.copy()
X_test_raw  = X_test.copy()

# scale (fit only on train)#标准化（只在训练集）并转回DataFrame
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns,index=X_train_raw.index)
X_val   = pd.DataFrame(scaler.transform(X_val), columns=X.columns,index=X_val_raw.index)
X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test_raw.index)
#scaler.transform 标准化 test 数据
#index=X_test_raw.index Pandas 会默认生成新的 0~n 索引
#加了 index=...，这样索引不会乱（对齐 y / 后面 debug 更稳）

# model (no internal early stopping, since you already have X_val)
model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=False,
    random_state=26
)

model.fit(X_train, y_train)
#测试集和预测集的预测
val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

# 训练集预测
train_pred = model.predict(X_train)
# 计算训练误差
train_mse = mean_squared_error(y_train, train_pred)

print("Baseline Train MSE:", train_mse)#加入是为了判断是不是欠拟合或者过拟合
print("Validation MSE:", mean_squared_error(y_val, val_pred))#Mean Squared Error（均方误差）
print("Test MSE:", mean_squared_error(y_test, test_pred))

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64
Shape of X: (20640, 8)
Missing values before cleaning:
MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64
Shape after removing missing values: (20640, 8)
Baseline Train MSE: 0.22777732804872386
Validation MSE: 0.28121556149977034
Test MSE: 0.24817189319758973


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
#检测训练集中是不是有出现多个样本经纬度完全一样
print("numbers of Duplicate coordinates：",X_train_raw[['Latitude','Longitude']].duplicated().sum())

duplicates = X_train_raw[X_train_raw[['Latitude','Longitude']].duplicated(keep=False)]

print(duplicates[['Latitude','Longitude']].sort_values(['Latitude','Longitude']))

X_train_raw.groupby(['Latitude','Longitude']).size().sort_values(ascending=False).head()
#训练集中确实有大量“经纬度完全相同”的样本
#对某个点来说，不止一个距离=0 的邻居，这会让 idxs = idxs[1:] 这种“删第一个当作自己”的逻辑不够严谨。
#只删掉第一个 idxs[1:]：可能删掉了“同坐标的另一个样本”,但仍然可能保留了自己的那条样本（如果它不是排在第一个）
#也可能保留多个 distance=0 的点（包含自己的 y），有轻微标签泄漏风险
#不能使用下列代码
'''
ArithmeticErrordef neighbour_mean_price(query_df, y_train_series, exclude_self=False):
    """
    为 query_df 的每个样本计算：训练集中k个邻居的 y 的平均值
    query_df: 需要计算邻居特征的数据（train/val/test 的 raw DataFrame）
    y_train_series: 训练集标签（只能用它，避免data leakage）
    exclude_self: 如果 query_df 是训练集本身，要剔除“自己”这个邻居
    """
    # indices：每个query点在训练集中对应的k个邻居的索引
    _, indices = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    neigh_mean = []  # 存放每个样本的邻居均价
    for idxs in indices:
        if exclude_self:
            # train 查询时，第一个邻居通常是自己，要去掉
            idxs = idxs[1:K+1]
        else:
            # val/test 直接取前K个邻居（它们全来自train）
            idxs = idxs[:K]
        # 用训练集的 y_train 取邻居价格并求平均（关键：不使用val/test真实y）
        neigh_mean.append(y_train_series.iloc[idxs].mean())

    return np.array(neigh_mean)
'''

# 1. 统计每个坐标出现多少次
coord_counts = X_train_raw.groupby(['Latitude', 'Longitude']).size()

# 2. 统计“出现次数”的分布
distribution = coord_counts.value_counts().sort_index()

# 3. 转成表格并加列名
distribution_df = distribution.reset_index()
distribution_df.columns = ['Repetition frequency of coordinate points', 'number of coordinate points']

# 4. 打印标题和表格
print("\n=== Repetitive statistics based on latitude and longitude ===")
print(distribution_df)

#用knn的局限性
#因为有非常多的坐标重复，所以会导致加入邻居特征时，在删除自己之后选择k个邻居选择到的领居出现不一样的情况。
#这种情况发生在和删除掉原坐标完全相同的坐标的情况非常少，在K值选择7的时候，有3个坐标，32个重复样本
#sklearn 在 多个距离相同的邻居之间没有唯一排序，返回的邻居集合可能因为内部实现顺序不同而变化
#用random seeds不能避免这种情况，random_state 只控制“随机过程”，只影响train_test_split，MLPRegressor 的权重初始化和可能的内部随机优化过程
#这种影响预测率的情况不是一定会发生，只是可能存在这种风险
#虽然代码每次运行其实返回的是同一组邻居（因为数据顺序不变，sklearn版本不变，算法参数不变），但sklearn实际上在距离相同的情况下就按内部顺序返回前 K 个
#有可能内部顺序改变一下可能预测准确率会提高或下降，所以我们找到的可能不是最优的参数解

'''
| 方法                 | 问题                |
| ------------------ | ----------------- |
| kNN                | 邻居数量固定，但邻居组合可能不唯一 |
| distance threshold | 邻居组合唯一，但邻居数量不固定   |
'''

numbers of Duplicate coordinates： 4789
       Latitude  Longitude
14759     32.56    -117.06
14760     32.56    -117.06
14753     32.56    -117.05
14754     32.56    -117.05
14758     32.56    -117.05
...         ...        ...
2591      40.88    -124.09
2618      40.93    -124.11
2617      40.93    -124.11
18837     41.73    -122.64
18836     41.73    -122.64

[7749 rows x 2 columns]

=== Repetitive statistics based on latitude and longitude ===
    Repetition frequency of coordinate points  number of coordinate points
0                                           1                         7111
1                                           2                         1836
2                                           3                          690
3                                           4                          269
4                                           5                          104
5                                           6                           35
6                       

In [ ]:
# ============================================================
# 第二个模型：加入 neighbourhood feature（NeighbourPrice）
# ============================================================

from sklearn.neighbors import NearestNeighbors  # 用于找k个最近邻

K = 7  # 邻居数量k（你们分支实验可改成3/5/10，但一次实验只改k）找到距离它最近的 7 套房子

# ------------------------------------------------------------
# 1) 用训练集的经纬度建立 kNN（只fit训练集）
#    这样 val/test 找邻居时，也只会在 train 里找（避免用到test标签）
# ------------------------------------------------------------
knn = NearestNeighbors(n_neighbors=K+1)  # 创建kNN搜索器
knn.fit(X_train_raw[['Latitude', 'Longitude']])  # 只用训练集坐标来fit

def neighbour_mean_price(query_df, y_train_series, exclude_self=False):
    """
    为 query_df 的每个样本计算：训练集中k个邻居的 y 的平均值
    query_df: 需要计算邻居特征的数据（train/val/test 的 raw DataFrame）
    y_train_series: 训练集标签（只能用它，避免data leakage）
    exclude_self: 如果 query_df 是训练集本身，要剔除“自己”这个邻居
    """

    #这行代码的作用是把 y_train_series 的顺序重新排列，使它与 X_train_raw 的索引顺序完全一致，从而保证每一行特征对应正确的标签。
    y_train_series = y_train_series.loc[X_train_raw.index]
    #根据每个样本的经纬度，在训练集中找到最近的 K 个邻居，并返回这些邻居的距离 (dists) 和在训练集中的位置索引 (inds)
    dists, inds = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    neigh_mean = [] ## 存放每个样本的邻居均价
    for row_i, idxs in enumerate(inds):
        if exclude_self:
            # 只剔除“自己这一行”的训练索引（最严谨），剔除自己（完全一样的那行数据）
            idxs = idxs[idxs != row_i]  # 用位置索引剔除自己
            idxs = idxs[:K]                 # 保证还是K个邻居，比如上一行代码要执行删除时，原数据并没有自己defensive programming（防御式编程）
            #比如以后用距离过滤，或者过滤多个点，不会变成 6 个或其他奇怪数量
        else:
            # val/test 直接取前K个邻居（它们全来自train）
            idxs = idxs[:K]
        # 用训练集的 y_train 取邻居价格并求平均（关键：不使用val/test真实y）
        neigh_mean.append(y_train_series.iloc[idxs].mean())

    return np.array(neigh_mean)
'''
觉得“我在训练集里拿某一条样本当 query，那 kNN 返回的邻居里怎么可能没有它自己？”
按数学定义，它应该一定在。
但你看到的 Case A（极少数）说明：在实现细节上，它“可能不在返回的前 K+1 个里”。
原因主要是 并列距离（tie）+ 只取前 K+1 个 + 内部取子集。

你并不是“先拿这一条样本再算平均”

你做的是：
用这条样本的经纬度作为 query
让 kneighbors() 在训练集里找 最接近的 n_neighbors 个点
返回这些点的索引
你再用这些索引去取 y_train 求平均

也就是说：
你并不是在邻居列表里手动把“自己”塞进去。
你完全依赖 kneighbors() 返回的那一组邻居
'''
# ------------------------------------------------------------
# 2) 计算 NeighbourPrice（邻居均价）
#    - train：剔除自己
#    - val/test：邻居都来自train，不存在“自己”，不用剔除
# ------------------------------------------------------------
neigh_price_train = neighbour_mean_price(X_train_raw, y_train, exclude_self=True)
neigh_price_val   = neighbour_mean_price(X_val_raw,   y_train, exclude_self=False)
neigh_price_test  = neighbour_mean_price(X_test_raw,  y_train, exclude_self=False)
#exclude_self=True 每个样本最近的点一定是自己，所以删除自己
#验证集样本根本不在训练集中，所以后面不需要删除自己

'''
| 变量          | 作用              |
| ----------- | --------------- |
| `X_val_raw` | 查询点（我想知道它附近房价）  |
| `y_train`   | 数据库（我用它来查邻居的价格） |
'''

# ------------------------------------------------------------
# 3) 把新特征train,val和test 的 neighbour price 加到 raw 数据里，做成新列（注意：从 raw 开始加）
# ------------------------------------------------------------
X_train_enh_raw = X_train_raw.copy()
X_val_enh_raw   = X_val_raw.copy()
X_test_enh_raw  = X_test_raw.copy()

X_train_enh_raw['NeighbourPrice'] = neigh_price_train
X_val_enh_raw['NeighbourPrice']   = neigh_price_val
X_test_enh_raw['NeighbourPrice']  = neigh_price_test

# ------------------------------------------------------------
# 4) 新增列后必须重新标准化（仍然只fit train）
# ------------------------------------------------------------
scaler2 = StandardScaler()
X_train_enh = pd.DataFrame(scaler2.fit_transform(X_train_enh_raw), columns=X_train_enh_raw.columns,index=X_train_enh_raw.index)
X_val_enh   = pd.DataFrame(scaler2.transform(X_val_enh_raw), columns=X_val_enh_raw.columns,index=X_val_enh_raw.index)
X_test_enh  = pd.DataFrame(scaler2.transform(X_test_enh_raw), columns=X_test_enh_raw.columns,index=X_test_enh_raw.index)

# ------------------------------------------------------------
# 5) 训练第二个模型（参数保持与baseline一致，保证公平对比）
# ------------------------------------------------------------
model_enh = MLPRegressor(
    hidden_layer_sizes=(64, 32),      # 与baseline相同
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=False,
    random_state=26
)

model_enh.fit(X_train_enh, y_train)

# ------------------------------------------------------------
# 6) 评估第二个模型
# ------------------------------------------------------------
val_pred_enh  = model_enh.predict(X_val_enh)
test_pred_enh = model_enh.predict(X_test_enh)

enh_val_mse  = mean_squared_error(y_val, val_pred_enh)
enh_test_mse = mean_squared_error(y_test, test_pred_enh)

train_pred_enh = model_enh.predict(X_train_enh)
enh_train_mse = mean_squared_error(y_train, train_pred_enh)

print("Enhanced Train MSE:", enh_train_mse)
print("Enhanced Validation MSE:", enh_val_mse)#模型在验证集上的平均平方误差是 0.222
print("Enhanced Test MSE:", enh_test_mse)#模型在最终测试集上的平均平方误差是 0.1835

# ------------------------------------------------------------
# 7) 和 baseline 对比（负数表示Enhanced更好）
# ------------------------------------------------------------
print("Delta Test MSE (Enhanced - Baseline):", enh_test_mse - mean_squared_error(y_test, test_pred))
#报告思路是我探索不同领域的参数a=，b=，c=

Enhanced Train MSE: 0.16227939588113074
Enhanced Validation MSE: 0.21618565935594575
Enhanced Test MSE: 0.17794248478639452
Delta Test MSE (Enhanced - Baseline): -0.07022940841119521


In [20]:
# 检查训练集中每个点的第一个邻居是否就是自己（通常为True）
d, ind = knn.kneighbors(X_train_raw[['Latitude','Longitude']])
print("Self in first neighbor:", np.mean(ind[:,0] == np.arange(len(X_train_raw))))

Self in first neighbor: 0.6777254374158815


In [21]:
#判断 训练集每个样本的 kneighbors 返回结果里到底有没有包含自己
import numpy as np

# 用你当前的 knn（已 fit 在 X_train_raw 上），以及当前 K
# 注意：这里检查的是训练集自身 query
dists, inds = knn.kneighbors(X_train_raw[['Latitude', 'Longitude']])

n = len(X_train_raw)

# 对训练集来说，“自己”的训练位置索引就是 0..n-1
self_pos = np.arange(n)

# A: 自己不在邻居列表里
self_in_list = (inds == self_pos[:, None]).any(axis=1)
count_A = np.sum(~self_in_list)

# B: 自己在邻居列表里，但不一定第一个
count_B = np.sum(self_in_list)

# B1: 自己在列表里且是第一个邻居
count_B_first = np.sum(inds[:, 0] == self_pos)

# B2: 自己在列表里但不是第一个
count_B_not_first = np.sum(self_in_list & (inds[:, 0] != self_pos))

print(f"Total train samples: {n}")
print(f"Case A (self NOT in returned neighbors): {count_A} ({count_A/n:.3%})")
print(f"Case B (self IN returned neighbors):     {count_B} ({count_B/n:.3%})")
print(f"  - B1: self is the 1st neighbor:        {count_B_first} ({count_B_first/n:.3%})")
print(f"  - B2: self NOT the 1st neighbor:       {count_B_not_first} ({count_B_not_first/n:.3%})")

# 给几个例子看看（最多打印5个）
if count_A > 0:
    ex = np.where(~self_in_list)[0][:5]
    print("\nExamples of Case A (row_i where self missing):", ex.tolist())
    print("Their neighbor indices (first row shown):")
    print(inds[ex[0]])
    print("Distances of that row:")
    print(dists[ex[0]])

if count_B_not_first > 0:
    ex = np.where(self_in_list & (inds[:, 0] != self_pos))[0][:5]
    print("\nExamples of Case B2 (self present but not first):", ex.tolist())
    r = ex[0]
    pos = np.where(inds[r] == r)[0][0]
    print(f"For row {r}, self appears at position {pos} in neighbor list.")
    print("Neighbor indices:", inds[r])
    print("Distances:", dists[r])

Total train samples: 14860
Case A (self NOT in returned neighbors): 8 (0.054%)
Case B (self IN returned neighbors):     14852 (99.946%)
  - B1: self is the 1st neighbor:        10071 (67.773%)
  - B2: self NOT the 1st neighbor:       4781 (32.174%)

Examples of Case A (row_i where self missing): [6173, 6873, 7904, 9156, 12306]
Their neighbor indices (first row shown):
[1707 6027 7238 5222 5032  740 5197 4263]
Distances of that row:
[0. 0. 0. 0. 0. 0. 0. 0.]

Examples of Case B2 (self present but not first): [1, 2, 4, 5, 18]
For row 1, self appears at position 1 in neighbor list.
Neighbor indices: [ 4713     1  7994 14772 13272 11203  7405  1659]
Distances: [0.   0.   0.   0.   0.   0.   0.01 0.01]
